# Biomedical Entity Extraction from Abstracts

This notebook extracts biomedical entities (diseases and chemicals/drugs) from abstract text and tracks how they appear and change over time. It is the first notebook to use the abstract text itself, so it runs on the local clean corpus only; the published metadata export contains no abstracts.

Two complementary approaches are used, for an honest reason. Proper biomedical NER with a trained model (scispaCy) gives high-quality entities but is too slow to run on all 3 million abstracts in a notebook (it would take many hours), so it is applied to a representative sample. A dictionary/gazetteer match against curated term lists is fast enough to run on every abstract, giving full-corpus coverage at lower recall. The dictionary provides the trends; the model provides a quality check and richer entity detail on the sample. Where they agree, confidence is high.

A flag, HAS_ABSTRACTS, guards the whole notebook so it skips cleanly if run against a metadata-only build.

## Setup

In [ ]:
import os, glob, collections, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: LOCAL ONLY (this notebook needs abstract text)
# =====================================================================
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")   # the clean corpus WITH abstracts

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "abstract", "abstract_len"])
df = df[df["year"] <= 2025].copy()

HAS_ABSTRACTS = bool((df["abstract"].fillna("").str.len() > 0).any())
print(f"loaded {len(df):,} records")
print("abstract text present:", HAS_ABSTRACTS)
if not HAS_ABSTRACTS:
    print("\nWARNING: no abstract text in this build. This notebook needs the local clean corpus.")
    print("The cells below will skip. Point DATA_DIR at a build that contains abstracts.")

## 1. Dictionary extraction (all abstracts)

A curated set of disease and chemical/drug terms is matched against every abstract by whole-word search. This is fast, transparent, and reproducible, but its recall is limited to the terms in the lists, so it undercounts entities not on them. The lists are explicit and can be extended. This pass gives full-corpus trends.

In [ ]:
# Curated term lists. These are seeds, not exhaustive; extend from the frequency output below.
DISEASE_TERMS = {
    "covid-19", "diabetes", "hypertension", "asthma", "cancer", "stroke", "obesity",
    "influenza", "pneumonia", "tuberculosis", "hiv", "sepsis", "depression", "anxiety",
    "alzheimer", "parkinson", "arthritis", "epilepsy", "leukemia", "melanoma",
    "covid", "sars-cov-2", "heart failure", "myocardial infarction", "atrial fibrillation",
}
CHEMICAL_TERMS = {
    "aspirin", "metformin", "insulin", "ibuprofen", "remdesivir", "warfarin", "statin",
    "heparin", "morphine", "paracetamol", "acetaminophen", "penicillin", "dexamethasone",
    "prednisone", "methotrexate", "tamoxifen", "lithium", "ketamine", "fluoxetine",
}

def build_matcher(vocab):
    """Compile ONE combined regex for the whole vocabulary (far faster than per-term search).
    Terms are sorted longest-first so multi-word terms match before their fragments."""
    terms_sorted = sorted(vocab, key=len, reverse=True)
    return re.compile(r"\b(" + "|".join(re.escape(t) for t in terms_sorted) + r")\b")

DISEASE_RE = build_matcher(DISEASE_TERMS)
CHEMICAL_RE = build_matcher(CHEMICAL_TERMS)

def dict_extract(text, matcher):
    """Return the unique vocabulary terms found in text (lowercased, whole-word)."""
    if not isinstance(text, str):
        return []
    return list(set(matcher.findall(text.lower())))

if HAS_ABSTRACTS:
    tqdm.pandas(desc="dictionary: diseases")
    df["dz_terms"] = df["abstract"].progress_map(lambda t: dict_extract(t, DISEASE_RE))
    tqdm.pandas(desc="dictionary: chemicals")
    df["chem_terms"] = df["abstract"].progress_map(lambda t: dict_extract(t, CHEMICAL_RE))

    dz_freq = collections.Counter(d for lst in df["dz_terms"] for d in lst)
    chem_freq = collections.Counter(c for lst in df["chem_terms"] for c in lst)
    print("top disease terms (dictionary, all abstracts):")
    for t, c in dz_freq.most_common(15):
        print(f"  {t:<25} {c:,}")
    print("\ntop chemical terms:")
    for t, c in chem_freq.most_common(10):
        print(f"  {t:<25} {c:,}")
else:
    print("skipped: no abstract text.")

**What this shows:** the most frequently mentioned diseases and chemicals across the corpus, by dictionary match. These counts are a lower bound (only listed terms are caught), but the relative ordering is informative and the per-abstract matches feed the trends below. Terms appearing unexpectedly high or low are a cue to extend or refine the lists.

## 2. Entity trends over time

Disease and chemical mentions per 1,000 abstracts per year, normalized so a rising line reflects genuine attention rather than corpus growth. This is the dictionary pass, so it covers all years.

In [ ]:
if HAS_ABSTRACTS:
    articles_per_year = df.groupby("year").size()

    def term_per_1000(col, term):
        has = df[col].map(lambda l: term in l)
        return (df.assign(h=has).groupby("year")["h"].sum() / articles_per_year * 1000)

    top_dz = [t for t, _ in dz_freq.most_common(8)]
    plt.figure(figsize=(13, 6))
    for term in top_dz:
        s = term_per_1000("dz_terms", term)
        plt.plot(s.index, s.values, marker="o", markersize=3, label=term)
    plt.title("Disease mentions over time (per 1,000 abstracts)")
    plt.xlabel("year"); plt.ylabel("per 1,000 abstracts")
    plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()
else:
    print("skipped: no abstract text.")

**What this shows:** the trajectory of each disease term as a share of abstracts. COVID-related terms should show the sharp post-2020 rise seen in notebook 07, while chronic conditions stay broadly stable. Because the rate is per 1,000 abstracts, rising lines indicate genuine growth in attention, not just more papers.

## 3. Proper NER with scispaCy (sample or full corpus)

For higher-quality extraction, a trained biomedical NER model (scispaCy's disease/chemical model) is run on a random sample. The sample keeps runtime to minutes while remaining representative. This requires `scispacy` and the `en_ner_bc5cdr_md` model; if they are not installed, this section skips with instructions and the notebook still completes on the dictionary results.

Install (run once, outside or in a cell):
`pip install scispacy` and the model from the scispaCy releases page (`en_ner_bc5cdr_md`).

The scope is controlled by `SAMPLE_N` in the next cell: a number runs a representative sample (fast, recommended), while `None` runs the full corpus (higher coverage but potentially many hours). Both results cache to separate files, so a sample run and a full run can coexist. The dictionary pass in sections 1-2 already covers the full corpus regardless of this setting.

In [ ]:
import os, sys, glob

# 1. Register the CUDA DLL directories FIRST (before any cupy/spacy import)
nvidia_root = os.path.join(sys.prefix, "Lib", "site-packages", "nvidia")
for bindir in glob.glob(os.path.join(nvidia_root, "*", "bin")):
    if os.path.isdir(bindir):
        os.add_dll_directory(bindir)
        os.environ["PATH"] = bindir + os.pathsep + os.environ.get("PATH", "")

# 2. Import and exercise cupy so it's confirmed working in THIS session
import cupy as cp
print("cupy works:", cp.array([1, 2, 3]).sum())   # must print 6

# 3. NOW import spacy and require GPU (thinc will see the working cupy)
import spacy
spacy.require_gpu()
print("GPU enabled")

In [ ]:
nlp = spacy.load("en_ner_bc5cdr_md", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
print("pipeline:", nlp.pipe_names)
doc = nlp("The patient was treated with aspirin for diabetes and hypertension.")
print([(e.text, e.label_) for e in doc.ents])

In [ ]:
from thinc.util import has_cupy
print("thinc sees cupy:", has_cupy)

In [ ]:
# SAMPLE_N controls the scispaCy NER pass:
#   SAMPLE_N = 50_000  -> run on a representative sample (fast, ~15-20 min, recommended)
#   SAMPLE_N = None    -> run on ALL abstracts (full corpus; can take many HOURS, see warning below)
# The dictionary pass (sections 1-2) always covers the full corpus regardless of this setting.
SAMPLE_N = 50_000

# Load the scispaCy biomedical NER model. `import scispacy` must come before the
# model load so its pipeline components are registered with spaCy.
nlp = None
if HAS_ABSTRACTS:
    try:
        import scispacy          # registers scispaCy components (required before load)
        import spacy
        nlp = spacy.load("en_ner_bc5cdr_md")     # disease + chemical NER
        print("scispaCy model loaded.")
    except Exception as e:
        print("scispaCy model not available; skipping the model pass.")
        print("To enable: pip install scispacy, then install en_ner_bc5cdr_md from the scispaCy releases.")
        print(f"(detail: {type(e).__name__}): {e}")

In [ ]:
import json

CACHE_DIR = os.path.join(ROOT, "data", "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Resolve scope: a number = sample of that size; None = the full corpus.
if SAMPLE_N is None:
    scope_n = len(df)
    cache_tag = "full"
    print(f"NER scope: FULL CORPUS ({scope_n:,} abstracts).")
    print("WARNING: this can take MANY HOURS at n_process=1. Consider running it once,")
    print("ideally overnight or with n_process raised, then rely on the cache afterwards.")
else:
    scope_n = min(SAMPLE_N, len(df))
    cache_tag = str(scope_n)
    print(f"NER scope: sample of {scope_n:,} abstracts.")

NER_CACHE = os.path.join(CACHE_DIR, f"ner_{cache_tag}.json")

# Select the rows to process. Full corpus = all rows; sample = fixed-seed sample.
if HAS_ABSTRACTS:
    if SAMPLE_N is None:
        sample = df
    else:
        sample = df.sample(scope_n, random_state=42)

if HAS_ABSTRACTS and os.path.exists(NER_CACHE):
    with open(NER_CACHE) as f:
        _p = json.load(f)
    ent_disease = collections.Counter(_p["disease"])
    ent_chemical = collections.Counter(_p["chemical"])
    # restore the exact rows used when the cache was built (sample mode stores indices)
    if "indices" in _p and _p["indices"] is not None:
        sample = df.loc[[i for i in _p["indices"] if i in df.index]]
    print(f"loaded cached NER results ({_p['n_sample']:,} abstracts). Delete {NER_CACHE} to recompute.")

elif HAS_ABSTRACTS and nlp is not None:
    # Abstracts are truncated to 1500 chars for the NER pass to keep runtime manageable.
    # Longer abstracts lose entities mentioned only in their final portion (noted in the caveats).
    texts = sample["abstract"].fillna("").str[:1500].tolist()

    ent_disease = collections.Counter()
    ent_chemical = collections.Counter()
    # n_process=1 is the safe default (n_process>1 can hang in Jupyter on Windows).
    # For the full corpus especially, raise n_process to your core count if it works on your machine.
    for doc in tqdm(nlp.pipe(texts, batch_size=128, n_process=1), total=len(texts), desc="scispaCy NER"):
        for e in doc.ents:
            label = e.label_.upper()
            term = e.text.lower().strip()
            if label == "DISEASE":
                ent_disease[term] += 1
            elif label == "CHEMICAL":
                ent_chemical[term] += 1

    # cache: store indices only for a sample (full corpus = all rows, indices not needed)
    payload = {"disease": dict(ent_disease), "chemical": dict(ent_chemical),
               "n_sample": len(sample),
               "indices": sample.index.tolist() if SAMPLE_N is not None else None}
    with open(NER_CACHE, "w") as f:
        json.dump(payload, f)
    print(f"NER done and cached to {NER_CACHE}")

    print(f"\nprocessed: {len(sample):,} abstracts")
    print(f"distinct disease entities: {len(ent_disease):,}")
    print(f"distinct chemical entities: {len(ent_chemical):,}")
    print("\ntop diseases (scispaCy):")
    for t, c in ent_disease.most_common(15):
        print(f"  {t:<30} {c:,}")
    print("\ntop chemicals (scispaCy):")
    for t, c in ent_chemical.most_common(15):
        print(f"  {t:<30} {c:,}")
else:
    print("scispaCy pass skipped (model not loaded or no abstracts).")

**What this shows:** the model pass finds far more distinct entities than the dictionary — 34,732 diseases and 18,249 chemicals in the sample alone. It recognises terms not on any list, including variants and rarer entities. `cancer` (5,823), `tumor` (5,001), and `pain` (3,103) top the disease list, while `alcohol` (1,796), `glucose` (1,322), and `smoking` (920) lead the chemicals. Because it runs on a sample, its counts are not directly comparable to the full‑corpus dictionary counts, but the ranking of common entities broadly agrees with the dictionary, which is the cross‑check.

Full run

In [ ]:
# SAMPLE_N controls the scispaCy NER pass:
#   SAMPLE_N = 50_000  -> run on a representative sample (fast, ~15-20 min, recommended)
#   SAMPLE_N = None    -> run on ALL abstracts (full corpus; can take many HOURS, see warning below)
# The dictionary pass (sections 1-2) always covers the full corpus regardless of this setting.
SAMPLE_N = None
# Load the scispaCy biomedical NER model. `import scispacy` must come before the
# model load so its pipeline components are registered with spaCy.
nlp = None
if HAS_ABSTRACTS:
    try:
        import scispacy          # registers scispaCy components (required before load)
        import spacy
        nlp = spacy.load("en_ner_bc5cdr_md")     # disease + chemical NER
        print("scispaCy model loaded.")
    except Exception as e:
        print("scispaCy model not available; skipping the model pass.")
        print("To enable: pip install scispacy, then install en_ner_bc5cdr_md from the scispaCy releases.")
        print(f"(detail: {type(e).__name__}): {e}")

In [ ]:
import json

CACHE_DIR = os.path.join(ROOT, "data", "cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Resolve scope: a number = sample of that size; None = the full corpus.
if SAMPLE_N is None:
    scope_n = len(df)
    cache_tag = "full"
    print(f"NER scope: FULL CORPUS ({scope_n:,} abstracts).")
    print("WARNING: this can take MANY HOURS at n_process=1. Consider running it once,")
    print("ideally overnight or with n_process raised, then rely on the cache afterwards.")
else:
    scope_n = min(SAMPLE_N, len(df))
    cache_tag = str(scope_n)
    print(f"NER scope: sample of {scope_n:,} abstracts.")

NER_CACHE = os.path.join(CACHE_DIR, f"ner_{cache_tag}.json")

# Select the rows to process. Full corpus = all rows; sample = fixed-seed sample.
if HAS_ABSTRACTS:
    if SAMPLE_N is None:
        sample = df
    else:
        sample = df.sample(scope_n, random_state=42)

if HAS_ABSTRACTS and os.path.exists(NER_CACHE):
    with open(NER_CACHE) as f:
        _p = json.load(f)
    ent_disease = collections.Counter(_p["disease"])
    ent_chemical = collections.Counter(_p["chemical"])
    # restore the exact rows used when the cache was built (sample mode stores indices)
    if "indices" in _p and _p["indices"] is not None:
        sample = df.loc[[i for i in _p["indices"] if i in df.index]]
    print(f"loaded cached NER results ({_p['n_sample']:,} abstracts). Delete {NER_CACHE} to recompute.")

elif HAS_ABSTRACTS and nlp is not None:
    # Abstracts are truncated to 1500 chars for the NER pass to keep runtime manageable.
    # Longer abstracts lose entities mentioned only in their final portion (noted in the caveats).
    texts = sample["abstract"].fillna("").str[:1500].tolist()

    ent_disease = collections.Counter()
    ent_chemical = collections.Counter()
    # n_process=1 is the safe default (n_process>1 can hang in Jupyter on Windows).
    # For the full corpus especially, raise n_process to your core count if it works on your machine.
    for doc in tqdm(nlp.pipe(texts, batch_size=512, n_process=1), total=len(texts), desc="scispaCy NER"):
        for e in doc.ents:
            label = e.label_.upper()
            term = e.text.lower().strip()
            if label == "DISEASE":
                ent_disease[term] += 1
            elif label == "CHEMICAL":
                ent_chemical[term] += 1

    # cache: store indices only for a sample (full corpus = all rows, indices not needed)
    payload = {"disease": dict(ent_disease), "chemical": dict(ent_chemical),
               "n_sample": len(sample),
               "indices": sample.index.tolist() if SAMPLE_N is not None else None}
    with open(NER_CACHE, "w") as f:
        json.dump(payload, f)
    print(f"NER done and cached to {NER_CACHE}")

    print(f"\nprocessed: {len(sample):,} abstracts")
    print(f"distinct disease entities: {len(ent_disease):,}")
    print(f"distinct chemical entities: {len(ent_chemical):,}")
    print("\ntop diseases (scispaCy):")
    for t, c in ent_disease.most_common(15):
        print(f"  {t:<30} {c:,}")
    print("\ntop chemicals (scispaCy):")
    for t, c in ent_chemical.most_common(15):
        print(f"  {t:<30} {c:,}")
else:
    print("scispaCy pass skipped (model not loaded or no abstracts).")

**What this shows:** the model pass finds far more distinct entities than the dictionary — 34,732 diseases and 18,249 chemicals in the sample alone. It recognises terms not on any list, including variants and rarer entities. `cancer` (5,823), `tumor` (5,001), and `pain` (3,103) top the disease list, while `alcohol` (1,796), `glucose` (1,322), and `smoking` (920) lead the chemicals. Because it runs on a sample, its counts are not directly comparable to the full‑corpus dictionary counts, but the ranking of common entities broadly agrees with the dictionary, which is the cross‑check.

## 4. Dictionary vs model agreement

Where the two methods agree on the most common entities, confidence is high; where they differ, it points to either dictionary gaps (terms the model found that the list lacks) or model noise. This compares the top entities from each on the sample.

In [ ]:
# Compare dictionary vs model on the same sample. Runs whenever NER results exist
# (freshly computed or loaded from cache).
if HAS_ABSTRACTS and "sample" in dir() and "ent_disease" in dir():
    samp_dz = collections.Counter(d for lst in sample["dz_terms"] for d in lst)

    dict_top = set(t for t, _ in samp_dz.most_common(20))
    model_top = set(t for t, _ in ent_disease.most_common(20))

    both = sorted(dict_top & model_top)
    only_model = sorted(model_top - dict_top)[:15]
    print("diseases in BOTH top-20 (high confidence):")
    print("  " + (", ".join(both) if both else "(none)"))
    print("\nfound by the MODEL but not in the dictionary top-20 (candidate list additions):")
    print("  " + (", ".join(only_model) if only_model else "(none)"))
else:
    print("comparison skipped (NER results not available).")

**What this shows:** the overlap between the two methods marks the entities both agree are common, the most trustworthy results. The terms the model finds but the dictionary misses are concrete suggestions for extending the term lists, turning the model pass into a tool for improving the fast dictionary pass.

## 5. Closing the loop: extending the dictionary with model discoveries

The model pass exists not only to extract entities but to improve the fast dictionary. This step takes the most frequent entities the model found that are absent from the seed lists, and proposes them as additions. This is the payoff of the dual-method design: the slow, high-recall model bootstraps the fast, full-corpus dictionary, so a future dictionary pass catches far more than the original hand-picked seeds.

The candidate terms below are filtered by frequency and basic noise removal, but they are proposals, not a finished vocabulary. They should be curated before use, because the model's recall comes with imprecision: it labels some symptoms and outcomes as diseases, and some physiological or behavioural terms as chemicals.

In [ ]:
# Use the model's full-corpus discoveries to extend the dictionary, then show the gain.
# Take frequent model entities not already in the dictionary (cleaned of obvious noise).
NOISE_ENT = {"±", "n", "pd", "i", "ii", "iii", "a", "b", "c", "+", "-", "/"}

model_disease_top = [t for t, c in ent_disease.most_common(100)
                     if t not in DISEASE_TERMS and t not in NOISE_ENT and len(t) > 2 and c >= 1000]
model_chemical_top = [t for t, c in ent_chemical.most_common(100)
                      if t not in CHEMICAL_TERMS and t not in NOISE_ENT and len(t) > 2 and c >= 1000]

print(f"model-discovered disease terms to add ({len(model_disease_top)}):")
print("  " + ", ".join(model_disease_top[:25]))
print(f"\nmodel-discovered chemical terms to add ({len(model_chemical_top)}):")
print("  " + ", ".join(model_chemical_top[:25]))

# the extended dictionary (curate this list before using in production)
DISEASE_TERMS_EXTENDED = DISEASE_TERMS | set(model_disease_top)
CHEMICAL_TERMS_EXTENDED = CHEMICAL_TERMS | set(model_chemical_top)
print(f"\ndictionary grew: diseases {len(DISEASE_TERMS)} -> {len(DISEASE_TERMS_EXTENDED)}, "
      f"chemicals {len(CHEMICAL_TERMS)} -> {len(CHEMICAL_TERMS_EXTENDED)}")

**What this shows:** the model surfaced 79 disease and 67 chemical candidates absent from the seed lists, growing the dictionary from 25 to 104 diseases and 19 to 86 chemicals. Many are clear, valuable additions the seeds missed: specific cancers (breast cancer, prostate cancer), psychiatric conditions (schizophrenia, PTSD, ADHD, dementia), and common substances (glucose, cholesterol, estrogen, dopamine).

The list also exposes the model's imprecision, which is itself an informative result. Several "disease" candidates are symptoms or outcomes rather than diseases (death, pain, trauma, bleeding, fatigue), and several "chemical" candidates are behaviours or physiological substances rather than drugs (smoking, alcohol, oxygen, calcium, sodium). This is expected from a high-recall NER model and is exactly why the candidates are proposals for curation, not an automatic vocabulary. A human pass should keep the genuine entities and drop the category-mismatched ones before any extended dictionary is used for analysis.

## 6. Summary and caveats

### What this notebook produced
This notebook extracted biomedical entities from abstract text two ways: a fast dictionary match across all abstracts (giving full-corpus disease and chemical trends), and a trained NER model (scispaCy) on a representative sample (giving higher-recall entities and a quality check). It reported the most common entities, their trends over time, and the agreement between the two methods.

### Caveats

**The dictionary pass is recall-limited by its term lists.** It only finds listed terms, so its counts are a lower bound and miss variants and rarer entities. The model-driven extension in section 5 mitigates this, but the original full-corpus trends were computed on the seed lists.

**The full NER pass stored aggregate counts, not per-year counts.** The cached results give total mentions per entity across the whole corpus, so model-based time trends are not available without re-running. Temporal trends in this notebook come from the dictionary pass, which is computed per year. The model's role here is recall (entity discovery), not trending.

**The model has high recall but imprecise category boundaries.** As section 5 shows, the BC5CDR model labels some symptoms and outcomes as diseases (death, pain, trauma) and some behaviours or physiological substances as chemicals (smoking, alcohol, oxygen). Its top entities should be curated before being treated as a clean disease or chemical vocabulary. Visible noise (single-character tokens, symbols such as the plus-or-minus sign) confirms the model is not error-free.

**The NER pass truncates abstracts to 1500 characters.** To keep runtime manageable, only the first 1500 characters of each abstract are processed, so entities mentioned only in the later portion of longer abstracts are missed in the model pass. The dictionary pass processes the full abstract text and is not truncated.

**Entity extraction is surface-level, not normalized.** Neither pass resolves synonyms or links entities to ontology identifiers, so "tumor" and "tumour", or "cancer" and "breast cancer", are counted separately. This is detection, not entity normalization; ontology linking (for example to UMLS or MeSH) would be a separate step.

**Abstracts only, and US-affiliated human-subject scope.** The text analysed is the abstract, not full text, so entities mentioned only in the body of a paper are missed. As with the rest of the corpus, results reflect the filtered scope.

---

## Notebook complete

This notebook extracted biomedical entities from abstract text two ways: a fast dictionary pass over all 3 million abstracts (full-corpus disease and chemical trends), and a trained scispaCy NER model run on both a 50,000-abstract sample and the full corpus (higher-recall entity discovery, GPU-accelerated and cached). It compared the two methods and used the model's discoveries to propose an extended dictionary.

The full-corpus NER found over 740,000 distinct disease mentions and 337,000 distinct chemical mentions, far beyond what the seed dictionary captured, confirming the value of the model pass for recall while the dictionary provides fast, reproducible full-corpus trends.

Natural extensions, each a separate piece of work: re-running the NER with per-year storage to obtain model-based temporal trends; linking entities to UMLS or MeSH for synonym resolution; and building disease-chemical co-occurrence networks from the entities detected here.

**Next up: Notebook 09 — re‑running the dictionary with the extended vocabulary**

Now that the NER model has discovered hundreds of new disease and chemical terms (Section 5), the next step is to put that extended dictionary to work.

The fast dictionary pass from Section 1 can be re‑run with the extended term lists (`DISEASE_TERMS_EXTENDED`, `CHEMICAL_TERMS_EXTENDED`). This takes only minutes (not hours), updates the full‑corpus trends with substantially improved recall, and preserves the per‑year normalisation that the NER cache (aggregate‑only) cannot provide.

This is the immediate payoff of the 5‑hour NER run: the model bootstraps the dictionary, and the dictionary delivers fast, full‑coverage temporal trends.

**Optional future work (if needed):** If exact NER spans are required over time, a separate run with per‑year caching would be needed. That would take another 5 hours and is not required for the core trend analysis presented here.